In [1]:
import copy
from pathlib import Path
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sunspot_features import FEATURE_SETS, build_features
import time
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit

In [2]:
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # before the first CUDA call
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

TRAIN_CSV = "../data_organizing/train_set_lon70.csv"
TEST_CSV = "../data_organizing/test_set_lon70.csv"
IC_CROPS_FOLDER = r"D:\IC-DeepLearning\cropped_data_v2"
MAG_CROPS_FOLDER = r"D:\IC-DeepLearning\cropped_data_v2_mag"
SUBSET_TAG = "lon70_det2"      
MAG_SCALE = 100.0
FEATURE_SET = "no_size"
SEEDS = [1, 2, 3, 4, 5]           
NUM_EPOCHS = 30
PATIENCE = 5
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TABULAR_COLUMNS = FEATURE_SETS[FEATURE_SET]
NUM_TABULAR = len(TABULAR_COLUMNS)

train_df = pd.read_csv(TRAIN_CSV, dtype={"nmbr": str})
test_df = pd.read_csv(TEST_CSV, dtype={"nmbr": str})

assert "mag_crop_filename" in train_df.columns and "mag_crop_filename" in test_df.columns, "not the lon70 files"
assert train_df["lon_mag"].abs().max() <= 70 and test_df["lon_mag"].abs().max() <= 70

classes = sorted(train_df["zurich_class"].unique())
class_to_idx = {cls: i for i, cls in enumerate(classes)}
NUM_CLASSES = len(class_to_idx)
print(class_to_idx, "| device:", DEVICE)
print("Feature set:", FEATURE_SET, "->", TABULAR_COLUMNS)

{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'H': 6} | device: cuda
Feature set: no_size -> ['lat', 'abs_lon', 'mag_simple', 'mag_mixed', 'mag_complex']


In [3]:
def make_tabular(df):
    return build_features(df)[TABULAR_COLUMNS].values.astype("float32")


def prepare_channels(ic, mag, stats):
    ic_mean, ic_std, mag_mean, mag_std = stats
    bad = np.isnan(ic) | np.isnan(mag)
    ic_n = (np.where(bad, ic_mean, ic) - ic_mean) / ic_std
    mag_n = (np.arcsinh(np.where(bad, 0.0, mag) / MAG_SCALE) - mag_mean) / mag_std
    ic_n[bad] = 0.0
    mag_n[bad] = 0.0
    return np.stack([ic_n, mag_n]).astype("float32")


class SunspotDataset(Dataset):
    def __init__(self, dataframe, ic_folder, mag_folder, class_to_idx, stats,
                 tab_mean, tab_std, augment=False, shuffle_mag=False):
        ok = dataframe["crop_filename"].notna() & dataframe["mag_crop_filename"].notna()
        self.data = dataframe[ok].reset_index(drop=True)
        self.ic_folder, self.mag_folder = Path(ic_folder), Path(mag_folder)
        self.class_to_idx = class_to_idx
        self.stats = stats
        self.augment = augment
        self.tabular = ((make_tabular(self.data) - tab_mean) / tab_std).astype("float32")

        # CONTROL EXPERIMENT: each sample receives the magnetogram of ANOTHER random sample of the same split
        self.shuffle_mag = shuffle_mag
        if shuffle_mag:
            n = len(self.data)
            perm = np.random.default_rng(123).permutation(n)
            same = perm == np.arange(n)
            perm[same] = (perm[same] + 1) % n          # nobody keeps its own magnetogram
            self.mag_perm = perm

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        ic = np.load(self.ic_folder / row["crop_filename"]).astype("float32")
        mag = np.load(self.mag_folder / row["mag_crop_filename"]).astype("float32")

        if self.shuffle_mag:
            other = self.data.iloc[self.mag_perm[idx]]
            mag_other = np.load(self.mag_folder / other["mag_crop_filename"]).astype("float32")
            # keep the REAL NaN mask, so the only thing that changes is the field content
            mag = np.where(np.isnan(mag), np.nan, np.nan_to_num(mag_other, nan=0.0)).astype("float32")

        image = torch.from_numpy(prepare_channels(ic, mag, self.stats))   # (2, 101, 101)

        if self.augment:
            if torch.rand(1).item() < 0.5:
                image = torch.flip(image, dims=[2])
            if torch.rand(1).item() < 0.5:
                image = torch.flip(image, dims=[1])

        tabular = torch.from_numpy(self.tabular[idx])
        label = self.class_to_idx[row["zurich_class"]]
        return image, tabular, label

In [4]:
train_df = train_df[train_df["crop_filename"].notna()].copy()
train_df["year"] = train_df["date"].str[:4]
groups = train_df["nmbr"].astype(str) + "_" + train_df["year"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(splitter.split(train_df, groups=groups))

train_split_df = train_df.iloc[train_idx]
val_split_df = train_df.iloc[val_idx]

print("Train rows:", len(train_split_df), "| Validation rows:", len(val_split_df))
print(val_split_df["zurich_class"].value_counts())

Train rows: 4450 | Validation rows: 777
zurich_class
D    224
C    152
H    126
B     93
E     82
A     70
F     30
Name: count, dtype: int64


In [5]:
# standardization statistics from the TRAINING split only
s, sq, n = np.zeros(2), np.zeros(2), 0
for fi, fm in zip(train_split_df["crop_filename"], train_split_df["mag_crop_filename"]):
    ic = np.load(Path(IC_CROPS_FOLDER) / fi).astype("float64")
    mag = np.load(Path(MAG_CROPS_FOLDER) / fm).astype("float64")
    valid = ~(np.isnan(ic) | np.isnan(mag))
    a, b = ic[valid], np.arcsinh(mag[valid] / MAG_SCALE)
    s += [a.sum(), b.sum()]
    sq += [(a ** 2).sum(), (b ** 2).sum()]
    n += valid.sum()

mean = s / n
std = np.sqrt(sq / n - mean ** 2)
STATS = (mean[0], std[0], mean[1], std[1])   # ic_mean, ic_std, mag_mean, mag_std
print("continuum mean/std:", STATS[0], STATS[1], "| magnetogram (arcsinh) mean/std:", STATS[2], STATS[3])

tab_train = make_tabular(train_split_df)
TAB_MEAN = tab_train.mean(axis=0)
TAB_STD = tab_train.std(axis=0)

continuum mean/std: 0.9964447120774987 0.04268916474176381 | magnetogram (arcsinh) mean/std: -0.008245825600744085 0.7170828013044397


In [7]:
train_dataset = SunspotDataset(train_split_df, IC_CROPS_FOLDER, MAG_CROPS_FOLDER, class_to_idx,
                               STATS, TAB_MEAN, TAB_STD, augment=True)
val_dataset = SunspotDataset(val_split_df, IC_CROPS_FOLDER, MAG_CROPS_FOLDER, class_to_idx,
                             STATS, TAB_MEAN, TAB_STD)
test_dataset = SunspotDataset(test_df, IC_CROPS_FOLDER, MAG_CROPS_FOLDER, class_to_idx,
                              STATS, TAB_MEAN, TAB_STD)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

assert train_dataset.tabular.shape[1] == NUM_TABULAR, train_dataset.tabular.shape
print("Tabular input shape:", train_dataset.tabular.shape)

Tabular input shape: (4450, 5)


In [8]:
# sanity check: no NaN/Inf and every crop is 1x101x101
def check_dataset(dataset, name):
    n_bad_values, n_bad_shape = 0, 0
    for idx in range(len(dataset)):
        image, tabular, _ = dataset[idx]
        if not (torch.isfinite(image).all() and torch.isfinite(tabular).all()):
            n_bad_values += 1
        if image.shape != (2, 101, 101):
            n_bad_shape += 1
    print(f"{name}: {len(dataset)} samples | non-finite values: {n_bad_values} | wrong shape: {n_bad_shape}")


check_dataset(train_dataset, "Train")
check_dataset(val_dataset, "Validation")
check_dataset(test_dataset, "Test")

Train: 4450 samples | non-finite values: 0 | wrong shape: 0
Validation: 777 samples | non-finite values: 0 | wrong shape: 0
Test: 630 samples | non-finite values: 0 | wrong shape: 0


In [10]:
# BUILDING THE MODELS
class SunspotHybridCNN(nn.Module):
    # use_image=False -> tabular-only control | use_tabular=False -> image-only CNN
    def __init__(self, num_classes, num_tabular=NUM_TABULAR, use_image=True, use_tabular=True, in_channels=1):
        super().__init__()
        self.use_image = use_image
        self.use_tabular = use_tabular

        self.conv1 = nn.Conv2d(in_channels, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2)
        self.adaptive_pool = nn.AvgPool2d(3)
        self.image_fc = nn.Linear(64 * 4 * 4, 128)

        self.tabular_fc = nn.Linear(num_tabular, 16)

        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(128 + 16, num_classes)

    def forward(self, image, tabular):
        if self.use_image:
            x = self.pool1(F.relu(self.conv1(image)))
            x = self.pool2(F.relu(self.conv2(x)))
            x = self.pool3(F.relu(self.conv3(x)))
            x = self.adaptive_pool(x).flatten(start_dim=1)
            image_features = self.dropout(F.relu(self.image_fc(x)))
        else:
            image_features = torch.zeros(image.size(0), 128, device=image.device)

        if self.use_tabular:
            tabular_features = F.relu(self.tabular_fc(tabular))
        else:
            tabular_features = torch.zeros(image.size(0), 16, device=image.device)

        return self.classifier(torch.cat([image_features, tabular_features], dim=1))


class SunspotResidualNet(nn.Module):
    def __init__(self, num_classes, num_tabular=NUM_TABULAR, use_image=True):
        super().__init__()
        self.use_image = use_image

        # tabular branch: produces the base logits
        self.tabular_fc = nn.Linear(num_tabular, 16)
        self.tabular_out = nn.Linear(16, num_classes)

        # image branch: only learns a correction on top of the base logits
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2)
        self.adaptive_pool = nn.AvgPool2d(3)
        self.image_fc = nn.Linear(64 * 4 * 4, 32)
        self.image_dropout = nn.Dropout(0.5)
        self.image_out = nn.Linear(32, num_classes)

        # zero init: training starts as the tabular-only model
        nn.init.zeros_(self.image_out.weight)
        nn.init.zeros_(self.image_out.bias)

    def forward(self, image, tabular):
        base_logits = self.tabular_out(F.relu(self.tabular_fc(tabular)))
        if not self.use_image:
            return base_logits

        x = self.pool1(F.relu(self.conv1(image)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        x = self.adaptive_pool(x).flatten(start_dim=1)
        x = self.image_dropout(F.relu(self.image_fc(x)))
        return base_logits + self.image_out(x)
    
MODEL_FACTORIES = {
    "tabular": lambda: SunspotHybridCNN(NUM_CLASSES, use_image=False, use_tabular=True),
    "image": lambda: SunspotHybridCNN(NUM_CLASSES, use_image=True, use_tabular=False),
    "hybrid": lambda: SunspotHybridCNN(NUM_CLASSES, use_image=True, use_tabular=True),
    "residual": lambda: SunspotResidualNet(NUM_CLASSES, use_image=True),
}
TAGS = ["image"]

MODEL_FACTORIES.update({
    "image": lambda: SunspotHybridCNN(NUM_CLASSES, in_channels=1, use_image=True, use_tabular=False),
    "image_mag": lambda: SunspotHybridCNN(NUM_CLASSES, in_channels=2, use_image=True, use_tabular=False),
})
CHANNELS_BY_TAG = {"image": 1, "image_mag": 2}
TAGS = ["image", "image_mag"]

In [11]:
WEIGHTS_DIR = Path(f"weights_{SUBSET_TAG}")   # separate folder: old weights (other data) are never reused
WEIGHTS_DIR.mkdir(exist_ok=True)


def weights_path_for(tag, seed):
    return WEIGHTS_DIR / f"best_{FEATURE_SET}_{tag}_seed{seed}.pt"


train_labels = train_dataset.data["zurich_class"].map(class_to_idx).values
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
class_weights = len(train_labels) / (NUM_CLASSES * class_counts)
criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
)

LOSS_BY_TAG = {}
def get_criterion(tag):
    return LOSS_BY_TAG.get(tag, criterion)

def weights_match_architecture(weights_path, tag):
    try:
        MODEL_FACTORIES[tag]().load_state_dict(torch.load(weights_path, map_location="cpu"))
        return True
    except Exception:
        return False


def train_one_seed(seed, tag):
    weights_path = weights_path_for(tag, seed)
    if weights_path.exists():
        if weights_match_architecture(weights_path, tag):
            print(f"[{tag}] seed {seed}: compatible weights found, skipping training")
            return
        print(f"[{tag}] seed {seed}: existing weights do not match this architecture, retraining")

    n_ch = CHANNELS_BY_TAG.get(tag, 1)
    start = time.time()

    torch.manual_seed(seed)
    model = MODEL_FACTORIES[tag]().to(DEVICE)
    # reseed AFTER building the model: image and image_mag then see the same batch order and flips
    torch.manual_seed(seed)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    best_f1, best_weights, epochs_without_improvement = -1.0, None, 0

    for epoch in range(NUM_EPOCHS):
        model.train()
        for images, tabular, labels in train_loader:
            images, tabular, labels = images[:, :n_ch].to(DEVICE), tabular.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = get_criterion(tag)(model(images, tabular), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, tabular, labels in val_loader:
                outputs = model(images[:, :n_ch].to(DEVICE), tabular.to(DEVICE))
                all_preds.extend(outputs.argmax(dim=1).cpu().tolist())
                all_labels.extend(labels.tolist())
        val_f1 = f1_score(all_labels, all_preds, average="macro")
        scheduler.step(val_f1)

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_weights = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                break

    torch.save(best_weights, weights_path)
    print(f"[{tag}] seed {seed}: best val macro F1 = {best_f1:.4f} "
          f"(stopped at epoch {epoch + 1}, {time.time() - start:.0f} s)")

In [10]:
for tag in TAGS:
    for seed in SEEDS:
        train_one_seed(seed, tag)

[image_mag] seed 1: best val macro F1 = 0.4888 (stopped at epoch 20, 168 s)
[image_mag] seed 2: best val macro F1 = 0.4712 (stopped at epoch 15, 124 s)
[image_mag] seed 3: best val macro F1 = 0.4955 (stopped at epoch 25, 208 s)
[image_mag] seed 4: best val macro F1 = 0.4896 (stopped at epoch 19, 158 s)
[image_mag] seed 5: best val macro F1 = 0.4904 (stopped at epoch 22, 184 s)


In [13]:
def predict_probs(model, loader, n_ch=1):
    model.eval()
    probs, labels_all = [], []
    with torch.no_grad():
        for images, tabular, labels in loader:
            logits = model(images[:, :n_ch].to(DEVICE), tabular.to(DEVICE))
            probs.append(F.softmax(logits, dim=1).cpu().numpy())
            labels_all.append(labels.numpy())
    return np.concatenate(probs), np.concatenate(labels_all)


def load_model(tag, seed):
    model = MODEL_FACTORIES[tag]().to(DEVICE)
    model.load_state_dict(torch.load(weights_path_for(tag, seed), map_location=DEVICE))
    return model


def evaluate_tag(tag, loader):
    n_ch = CHANNELS_BY_TAG.get(tag, 1)
    seed_probs = []
    for seed in SEEDS:
        probs, labels = predict_probs(load_model(tag, seed), loader, n_ch)
        seed_probs.append(probs)
    all_classes = list(range(NUM_CLASSES))
    per_seed_f1 = np.array([f1_score(labels, p.argmax(axis=1), average="macro") for p in seed_probs])
    per_seed_class_f1 = np.array([f1_score(labels, p.argmax(axis=1), average=None, labels=all_classes)
                                  for p in seed_probs])              # (n_seeds, n_classes)
    ensemble_preds = np.mean(seed_probs, axis=0).argmax(axis=1)
    return {"per_seed_f1": per_seed_f1, "per_seed_class_f1": per_seed_class_f1,
            "labels": labels, "ensemble_preds": ensemble_preds}


val_results = {tag: evaluate_tag(tag, val_loader) for tag in TAGS}

rows = []
for tag, res in val_results.items():
    rows.append({
        "model": tag,
        "mean macro F1": res["per_seed_f1"].mean(),
        "std (seeds)": res["per_seed_f1"].std(ddof=1),
        "ensemble macro F1": f1_score(res["labels"], res["ensemble_preds"], average="macro"),
    })
print(pd.DataFrame(rows).round(4).to_string(index=False))


per_class = {tag: f1_score(r["labels"], r["ensemble_preds"], average=None, labels=list(range(NUM_CLASSES)))
             for tag, r in val_results.items()}
print()
print(pd.DataFrame(per_class, index=list(class_to_idx)).round(2).to_string())

    model  mean macro F1  std (seeds)  ensemble macro F1
    image         0.4903       0.0043             0.5034
image_mag         0.5238       0.0055             0.5376

   image  image_mag
A   0.50       0.51
B   0.28       0.39
C   0.31       0.33
D   0.66       0.65
E   0.62       0.66
F   0.52       0.56
H   0.64       0.66


In [14]:
test_res = {tag: evaluate_tag(tag, test_loader) for tag in ("image", "image_mag")}
a, b = test_res["image"], test_res["image_mag"]

print("TEST 2005 macro F1 per seed  image    :", a["per_seed_f1"].round(3), "| mean %.3f std %.3f" % (a["per_seed_f1"].mean(), a["per_seed_f1"].std(ddof=1)))
print("TEST 2005 macro F1 per seed  image_mag:", b["per_seed_f1"].round(3), "| mean %.3f std %.3f" % (b["per_seed_f1"].mean(), b["per_seed_f1"].std(ddof=1)))
diff = b["per_seed_f1"] - a["per_seed_f1"]
print("paired difference per seed:", diff.round(3), "| mean %.3f" % diff.mean())

rows = []
for i, cls in enumerate(class_to_idx):
    d = b["per_seed_class_f1"][:, i] - a["per_seed_class_f1"][:, i]
    rows.append({"class": cls, "n_test": int((a["labels"] == i).sum()),
                 "image": a["per_seed_class_f1"][:, i].mean(), "image_mag": b["per_seed_class_f1"][:, i].mean(),
                 "diff": d.mean(), "std of diff": d.std(ddof=1)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

TEST 2005 macro F1 per seed  image    : [0.433 0.433 0.461 0.447 0.458] | mean 0.446 std 0.013
TEST 2005 macro F1 per seed  image_mag: [0.423 0.438 0.461 0.465 0.472] | mean 0.452 std 0.021
paired difference per seed: [-0.01   0.005  0.     0.018  0.014] | mean 0.005
class  n_test  image  image_mag   diff  std of diff
    A      37  0.439      0.448  0.010        0.017
    B      36  0.175      0.224  0.049        0.024
    C     114  0.372      0.382  0.010        0.057
    D     225  0.624      0.625  0.002        0.019
    E      74  0.509      0.524  0.015        0.025
    F      14  0.412      0.369 -0.043        0.035
    H     130  0.593      0.589 -0.004        0.054


In [13]:
res = evaluate_tag("image_mag", val_loader)
print("shuffled-magnetogram control, macro F1 per seed:", res["per_seed_f1"].round(3),
      "| mean %.3f std %.3f" % (res["per_seed_f1"].mean(), res["per_seed_f1"].std(ddof=1)))

ref = pd.DataFrame({   # numbers from the reproducible run (real magnetogram vs continuum only)
    "image":      [0.480, 0.268, 0.320, 0.647, 0.606, 0.476, 0.634],
    "image_mag":  [0.515, 0.405, 0.328, 0.643, 0.626, 0.520, 0.631]}, index=list(class_to_idx))
ref["shuffled"] = res["per_seed_class_f1"].mean(axis=0)
print("\nper-class F1 (mean over 5 seeds); reference: image 0.490, image_mag real 0.524")
print(ref.round(3))

shuffled-magnetogram control, macro F1 per seed: [0.489 0.471 0.495 0.49  0.49 ] | mean 0.487 std 0.009

per-class F1 (mean over 5 seeds); reference: image 0.490, image_mag real 0.524
   image  image_mag  shuffled
A  0.480      0.515     0.490
B  0.268      0.405     0.260
C  0.320      0.328     0.321
D  0.647      0.643     0.629
E  0.606      0.626     0.588
F  0.476      0.520     0.484
H  0.634      0.631     0.638


In [14]:
val_results = {tag: evaluate_tag(tag, val_loader) for tag in TAGS}

a, b = val_results["image"], val_results["image_mag"]
print("macro F1 per seed  image    :", a["per_seed_f1"].round(3), "| mean %.3f std %.3f" % (a["per_seed_f1"].mean(), a["per_seed_f1"].std(ddof=1)))
print("macro F1 per seed  image_mag:", b["per_seed_f1"].round(3), "| mean %.3f std %.3f" % (b["per_seed_f1"].mean(), b["per_seed_f1"].std(ddof=1)))

diff = b["per_seed_f1"] - a["per_seed_f1"]
print("paired difference (image_mag - image) per seed:", diff.round(3), "| mean %.3f" % diff.mean())

rows = []
for i, cls in enumerate(class_to_idx):
    d = b["per_seed_class_f1"][:, i] - a["per_seed_class_f1"][:, i]
    rows.append({"class": cls, "n_val": int((a["labels"] == i).sum()),
                 "image": a["per_seed_class_f1"][:, i].mean(), "image_mag": b["per_seed_class_f1"][:, i].mean(),
                 "diff": d.mean(), "std of diff": d.std(ddof=1)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

macro F1 per seed  image    : [0.49  0.484 0.49  0.493 0.495] | mean 0.490 std 0.004
macro F1 per seed  image_mag: [0.526 0.528 0.529 0.517 0.52 ] | mean 0.524 std 0.005
paired difference (image_mag - image) per seed: [0.036 0.044 0.039 0.024 0.024] | mean 0.034
class  n_val  image  image_mag   diff  std of diff
    A     70  0.480      0.515  0.035        0.032
    B     93  0.268      0.405  0.136        0.054
    C    152  0.320      0.328  0.007        0.028
    D    224  0.647      0.643 -0.004        0.010
    E     82  0.606      0.626  0.020        0.024
    F     30  0.476      0.520  0.043        0.117
    H    126  0.634      0.631 -0.004        0.076


In [ ]:
def within_one_class(labels, preds):
    # share of A..F samples predicted as the exact class or an adjacent one (H is outside the A-F scale)
    rank = {class_to_i
    dx[c]: pos for pos, c in enumerate("ABCDEF")}
    in_scale = [(t, p) for t, p in zip(labels, preds) if t in rank]
    hits = sum(1 for t, p in in_scale if p in rank and abs(rank[t] - rank[p]) <= 1)
    return hits / len(in_scale)


# tabular baselines, same standardized features and training split as the networks
X_tr, y_tr = train_dataset.tabular, train_dataset.data["zurich_class"].map(class_to_idx).values
X_te, y_te = test_dataset.tabular, test_dataset.data["zurich_class"].map(class_to_idx).values

logreg = LogisticRegression(class_weight="balanced", solver="lbfgs", max_iter=1000).fit(X_tr, y_tr)
dummy = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)

test_results = {tag: evaluate_tag(tag, test_loader) for tag in TAGS}
for res in test_results.values():
    assert np.array_equal(res["labels"], y_te), "label order mismatch between loader and dataframe"

predictions = {
    "dummy": dummy.predict(X_te),
    "logistic regression": logreg.predict(X_te),
    **{tag: res["ensemble_preds"] for tag, res in test_results.items()},
}

rows = []
for name, preds in predictions.items():
    seed_f1 = test_results[name]["per_seed_f1"] if name in test_results else None
    rows.append({
        "model": name,
        "macro F1": f1_score(y_te, preds, average="macro"),
        "accuracy": accuracy_score(y_te, preds),
        "within one class (A-F)": within_one_class(y_te, preds),
        "mean seed F1": seed_f1.mean() if seed_f1 is not None else np.nan,
        "std seed F1": seed_f1.std(ddof=1) if seed_f1 is not None else np.nan,
    })
final_table = pd.DataFrame(rows).round(4)
print(final_table.to_string(index=False))
final_table.to_csv(f"final_test_results_{FEATURE_SET}.csv", index=False)

per_class_test = pd.DataFrame(
    {name: f1_score(y_te, preds, average=None, labels=list(range(NUM_CLASSES)))
     for name, preds in predictions.items()},
    index=list(class_to_idx),
).round(2)
print("\nPer-class F1 on the test set:")
print(per_class_test.to_string())

              model  macro F1  accuracy  within one class (A-F)  mean seed F1  std seed F1
              dummy    0.0713    0.3326                  0.8140           NaN          NaN
logistic regression    0.1600    0.2051                  0.4298           NaN          NaN
            tabular    0.1691    0.2179                  0.4890        0.1738       0.0123
              image    0.4317    0.5016                  0.7755        0.4223       0.0282
             hybrid    0.4615    0.5303                  0.7961        0.4484       0.0308
           residual    0.4198    0.4963                  0.7727        0.4250       0.0299

Per-class F1 on the test set:
   dummy  logistic regression  tabular  image  hybrid  residual
A    0.0                 0.08     0.07   0.47    0.48      0.47
B    0.0                 0.09     0.09   0.21    0.21      0.17
C    0.0                 0.12     0.13   0.37    0.41      0.35
D    0.5                 0.00     0.00   0.58    0.61      0.59
E    0.0    

In [13]:
# detailed report for the pure CNN (the research question) and for the best network chosen on VALIDATION
best_tag = max(val_results, key=lambda t: f1_score(val_results[t]["labels"],
                                                   val_results[t]["ensemble_preds"], average="macro"))
print("Best network on validation:", best_tag)

for tag in dict.fromkeys(["image", best_tag]):
    preds = test_results[tag]["ensemble_preds"]
    print(f"\n===== {tag} (ensemble of {len(SEEDS)} seeds) | test 2005 =====")
    print(classification_report(y_te, preds, target_names=list(class_to_idx), digits=3))
    cm = confusion_matrix(y_te, preds, labels=list(range(NUM_CLASSES)))
    print(pd.DataFrame(cm, index=list(class_to_idx), columns=list(class_to_idx)))

Best network on validation: hybrid

===== image (ensemble of 5 seeds) | test 2005 =====
              precision    recall  f1-score   support

           A      0.338     0.781     0.472        64
           B      0.173     0.275     0.212        51
           C      0.404     0.346     0.373       182
           D      0.714     0.495     0.585       313
           E      0.542     0.542     0.542        96
           F      0.286     0.200     0.235        20
           H      0.585     0.623     0.604       215

    accuracy                          0.502       941
   macro avg      0.434     0.466     0.432       941
weighted avg      0.543     0.502     0.508       941

    A   B   C    D   E  F    H
A  50   8   1    0   0  0    5
B  26  14   7    2   0  1    1
C  18  22  63   24   1  1   53
D  12  23  58  155  31  0   34
E   0   1   4   32  52  6    1
F   0   1   2    1  11  4    1
H  42  12  21    3   1  2  134

===== hybrid (ensemble of 5 seeds) | test 2005 =====
             

In [14]:
class FocalLoss(nn.Module):
    # CE * (1 - p_t)^gamma : down-weights easy samples
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, targets):
        log_pt = F.log_softmax(logits, dim=1).gather(1, targets.unsqueeze(1)).squeeze(1)
        loss = -((1 - log_pt.exp()) ** self.gamma) * log_pt
        if self.weight is None:
            return loss.mean()
        w = self.weight[targets]
        return (loss * w).sum() / w.sum()  # same normalization as nn.CrossEntropyLoss(weight=...)


class OrdinalLogLoss(nn.Module):
    # L = - sum_k d(y, k)^alpha * log(1 - p_k)   (Castagnos et al., 2022)
    def __init__(self, cost_matrix, alpha=1.0, weight=None):
        super().__init__()
        self.cost = cost_matrix ** alpha
        self.weight = weight

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1).clamp(max=1 - 1e-6)
        loss = -(torch.log1p(-probs) * self.cost[targets]).sum(dim=1)
        if self.weight is None:
            return loss.mean()
        w = self.weight[targets]
        return (loss * w).sum() / w.sum()


def build_cost_matrix(h_cost=2.0):
    # |rank difference| inside A..F; H is outside the scale, so it gets a constant cost to everything
    rank = {class_to_idx[c]: pos for pos, c in enumerate("ABCDEF")}
    cost = torch.zeros(NUM_CLASSES, NUM_CLASSES)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            if i == j:
                continue
            cost[i, j] = abs(rank[i] - rank[j]) if (i in rank and j in rank) else h_cost
    return cost.to(DEVICE)


print(build_cost_matrix().cpu().numpy())

[[0. 1. 2. 3. 4. 5. 2.]
 [1. 0. 1. 2. 3. 4. 2.]
 [2. 1. 0. 1. 2. 3. 2.]
 [3. 2. 1. 0. 1. 2. 2.]
 [4. 3. 2. 1. 0. 1. 2.]
 [5. 4. 3. 2. 1. 0. 2.]
 [2. 2. 2. 2. 2. 2. 0.]]


In [15]:
LOSS_WEIGHTS = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)  # None -> favors accuracy over macro F1
COST = build_cost_matrix()

LOSS_MAKERS = {
    "focal": lambda: FocalLoss(gamma=2.0, weight=LOSS_WEIGHTS),
    "oll": lambda: OrdinalLogLoss(COST, alpha=1.0, weight=LOSS_WEIGHTS),
}
ARCHS = {"hybrid": MODEL_FACTORIES["hybrid"], "residual": MODEL_FACTORIES["residual"]}

ALT_TAGS = []
for arch, factory in ARCHS.items():
    for loss_name, make_loss in LOSS_MAKERS.items():
        tag = f"{arch}_{loss_name}"
        MODEL_FACTORIES[tag] = factory      # same architecture as the original model
        LOSS_BY_TAG[tag] = make_loss()
        ALT_TAGS.append(tag)
print(ALT_TAGS)

['hybrid_focal', 'hybrid_oll', 'residual_focal', 'residual_oll']


In [16]:
for tag in ALT_TAGS:
    for seed in SEEDS:
        train_one_seed(seed, tag)

[hybrid_focal] seed 1: best val macro F1 = 0.5103 (stopped at epoch 15)
[hybrid_focal] seed 2: best val macro F1 = 0.5276 (stopped at epoch 20)
[hybrid_focal] seed 3: best val macro F1 = 0.5136 (stopped at epoch 11)
[hybrid_focal] seed 4: best val macro F1 = 0.5121 (stopped at epoch 12)
[hybrid_focal] seed 5: best val macro F1 = 0.5099 (stopped at epoch 11)
[hybrid_oll] seed 1: best val macro F1 = 0.5143 (stopped at epoch 23)
[hybrid_oll] seed 2: best val macro F1 = 0.5291 (stopped at epoch 19)
[hybrid_oll] seed 3: best val macro F1 = 0.4999 (stopped at epoch 21)
[hybrid_oll] seed 4: best val macro F1 = 0.5177 (stopped at epoch 28)
[hybrid_oll] seed 5: best val macro F1 = 0.5239 (stopped at epoch 30)
[residual_focal] seed 1: best val macro F1 = 0.5301 (stopped at epoch 25)
[residual_focal] seed 2: best val macro F1 = 0.4944 (stopped at epoch 15)
[residual_focal] seed 3: best val macro F1 = 0.4869 (stopped at epoch 12)
[residual_focal] seed 4: best val macro F1 = 0.5099 (stopped at epoc

In [17]:
alt_val_results = {tag: evaluate_tag(tag, val_loader) for tag in ALT_TAGS}

rows = []
for tag, res in {**{t: val_results[t] for t in ("hybrid", "residual")}, **alt_val_results}.items():
    rows.append({
        "model": tag,
        "mean macro F1": res["per_seed_f1"].mean(),
        "std (seeds)": res["per_seed_f1"].std(ddof=1),
        "ensemble macro F1": f1_score(res["labels"], res["ensemble_preds"], average="macro"),
        "ensemble accuracy": accuracy_score(res["labels"], res["ensemble_preds"]),
    })
print(pd.DataFrame(rows).round(4).to_string(index=False))

         model  mean macro F1  std (seeds)  ensemble macro F1  ensemble accuracy
        hybrid         0.5161       0.0103             0.5283             0.5448
      residual         0.5102       0.0141             0.5107             0.5206
  hybrid_focal         0.5147       0.0074             0.5223             0.5278
    hybrid_oll         0.5170       0.0111             0.5270             0.5238
residual_focal         0.5029       0.0174             0.5239             0.5303
  residual_oll         0.4953       0.0209             0.5123             0.5044


In [18]:
best_alt = max(alt_val_results, key=lambda t: accuracy_score(alt_val_results[t]["labels"],
                                                             alt_val_results[t]["ensemble_preds"]))
print("Best alternative-loss model on validation:", best_alt)

res = evaluate_tag(best_alt, test_loader)
preds = res["ensemble_preds"]
base_arch = best_alt.split("_")[0]

print(f"\n{best_alt}: accuracy {accuracy_score(y_te, preds):.4f} | macro F1 {f1_score(y_te, preds, average='macro'):.4f} "
      f"| within one class {within_one_class(y_te, preds):.4f}")
print(f"{base_arch} (cross-entropy): accuracy {accuracy_score(y_te, test_results[base_arch]['ensemble_preds']):.4f} "
      f"| macro F1 {f1_score(y_te, test_results[base_arch]['ensemble_preds'], average='macro'):.4f}")

print(classification_report(y_te, preds, target_names=list(class_to_idx), digits=3))
cm = confusion_matrix(y_te, preds, labels=list(range(NUM_CLASSES)))
print(pd.DataFrame(cm, index=list(class_to_idx), columns=list(class_to_idx)))

Best alternative-loss model on validation: residual_focal

residual_focal: accuracy 0.4846 | macro F1 0.3977 | within one class 0.7755
residual (cross-entropy): accuracy 0.4963 | macro F1 0.4198
              precision    recall  f1-score   support

           A      0.336     0.625     0.437        64
           B      0.135     0.196     0.160        51
           C      0.395     0.330     0.359       182
           D      0.701     0.479     0.569       313
           E      0.452     0.542     0.493        96
           F      0.167     0.150     0.158        20
           H      0.566     0.656     0.608       215

    accuracy                          0.485       941
   macro avg      0.393     0.425     0.398       941
weighted avg      0.519     0.485     0.490       941

    A   B   C    D   E   F    H
A  40  14   2    0   0   0    8
B  26  10   9    2   1   0    3
C  17  20  60   29   2   0   54
D   6  16  52  150  44   5   40
E   0   1   1   30  52  10    2
F   0   0   1   

In [20]:
# sanity check: which inputs and which weights are really being used?
print("FEATURE_SET:", FEATURE_SET)
print("TABULAR_COLUMNS:", TABULAR_COLUMNS)           # expected: lat, abs_lon, mag_simple, mag_mixed, mag_complex
print("Tabular shape:", train_dataset.tabular.shape)  # expected: (6951, 5)
print(sorted(p.name for p in WEIGHTS_DIR.glob("best_*seed0.pt")))  # expected: names containing "no_size"

FEATURE_SET: no_size
TABULAR_COLUMNS: ['lat', 'abs_lon', 'mag_simple', 'mag_mixed', 'mag_complex']
Tabular shape: (6951, 5)
[]
